In [1]:
import gc
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib

pd.set_option('display.float_format', '{:.4f}'.format)

In [2]:
TRAIN_PATH      = 'train_solo_track.parquet'
TEST_PATH       = 'test_solo_track.parquet'
SUBMISSION_PATH = 'submission_v7.csv'
TARGET_COL      = 'target_1h'
FORECAST_POINTS = 8
RANDOM_STATE    = 42

ENSEMBLE_OFFSETS = [0, 1, 2, 3]
ENSEMBLE_WEIGHTS = [0.4, 0.3, 0.2, 0.1]

LGBM_PARAMS = dict(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=127,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_samples=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

RU_HOLIDAYS = pd.to_datetime([
    '2025-01-01', '2025-01-02', '2025-01-03', '2025-01-07',
    '2025-02-24', '2025-03-10', '2025-05-01', '2025-05-02',
    '2025-05-09', '2025-06-12', '2025-11-04',
])
HOLIDAY_SET = set(RU_HOLIDAYS.normalize())

In [3]:
print('Загружаем данные...')
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])
test_df['timestamp']  = pd.to_datetime(test_df['timestamp'])
train_df = train_df.sort_values(['route_id', 'timestamp']).reset_index(drop=True)
for col in train_df.select_dtypes('float64').columns:
    train_df[col] = train_df[col].astype('float32')
print('Train shape:', train_df.shape)

Загружаем данные...
Train shape: (4630000, 9)


In [4]:
def build_features(df, step=None):
    d = df.copy()
    grp = d.groupby('route_id', sort=False)[TARGET_COL]

    for s in range(1, FORECAST_POINTS + 1):
        d[f'target_step_{s}'] = grp.shift(-s).astype('float32')

    for lag in [1, 2, 3, 4, 6, 8, 12, 16, 48, 336]:
        d[f'lag_{lag}'] = grp.shift(lag).astype('float32')

    if step is not None:
        d['lag_step']      = grp.shift(step).astype('float32')
        d['lag_step_week'] = grp.shift(336 + step).astype('float32')
        d['lag_step_2w']   = grp.shift(672 + step).astype('float32')

    shifted = grp.shift(1)
    for window in [4, 8, 48]:
        d[f'rolling_mean_{window}'] = shifted.rolling(window).mean().values.astype('float32')
        d[f'rolling_std_{window}']  = shifted.rolling(window).std().values.astype('float32')

    d['hour']         = d['timestamp'].dt.hour.astype('int8')
    d['minute']       = d['timestamp'].dt.minute.astype('int8')
    d['day_of_week']  = d['timestamp'].dt.dayofweek.astype('int8')
    d['is_weekend']   = (d['day_of_week'] >= 5).astype('int8')
    d['day_of_month'] = d['timestamp'].dt.day.astype('int8')
    d['hour_sin']     = np.sin(2 * np.pi * d['hour'] / 24).astype('float32')
    d['hour_cos']     = np.cos(2 * np.pi * d['hour'] / 24).astype('float32')
    d['dow_sin']      = np.sin(2 * np.pi * d['day_of_week'] / 7).astype('float32')
    d['dow_cos']      = np.cos(2 * np.pi * d['day_of_week'] / 7).astype('float32')
    d['is_holiday']      = d['timestamp'].dt.normalize().isin(HOLIDAY_SET).astype('int8')
    d['is_holiday_next'] = (d['timestamp'] + pd.Timedelta(days=1)).dt.normalize().isin(HOLIDAY_SET).astype('int8')
    d['is_holiday_prev'] = (d['timestamp'] - pd.Timedelta(days=1)).dt.normalize().isin(HOLIDAY_SET).astype('int8')

    status_cols         = sorted([c for c in d.columns if c.startswith('status_')])
    current_status_cols = [c for c in status_cols if int(c.split('_')[1]) <= 3]
    prev_status_cols    = [c for c in status_cols if int(c.split('_')[1]) >  3]
    d['status_current_sum'] = d[current_status_cols].sum(axis=1).astype('float32')
    d['status_prev_sum']    = d[prev_status_cols].sum(axis=1).astype('float32')
    d['status_ratio']       = (d['status_current_sum'] / (d['status_current_sum'] + d['status_prev_sum'] + 1)).astype('float32')
    grp_s = d.groupby('route_id', sort=False)
    for col in ['status_current_sum', 'status_prev_sum']:
        d[f'{col}_roll4'] = grp_s[col].shift(1).rolling(4).mean().values.astype('float32')

    route_agg = d.groupby('route_id')[TARGET_COL].agg(
        route_mean='mean', route_std='std',
        route_p10=lambda x: x.quantile(0.1),
        route_p90=lambda x: x.quantile(0.9)
    ).astype('float32')
    d = d.join(route_agg, on='route_id')
    d['target_norm']    = (d[TARGET_COL] / (d['route_mean'] + 1)).astype('float32')
    d['diff_from_mean'] = (d[TARGET_COL] - d['route_mean']).astype('float32')
    rh_mean = d.groupby(['route_id', 'hour'])[TARGET_COL].mean().rename('route_hour_mean').astype('float32')
    d = d.join(rh_mean, on=['route_id', 'hour'])
    d['vs_route_hour'] = (d[TARGET_COL] / (d['route_hour_mean'] + 1)).astype('float32')

    return d

print('build_features готова')

build_features готова


In [5]:
# ── Обучение отдельной модели на каждый step ──────────────────────────────────
FUTURE_TARGET_COLS = [f'target_step_{s}' for s in range(1, FORECAST_POINTS + 1)]
exclude_base       = {'timestamp', 'id', TARGET_COL} | set(FUTURE_TARGET_COLS)
cat_features_base  = ['route_id', 'hour', 'minute', 'day_of_week', 'is_weekend', 'day_of_month']

models                = {}
bias_factors          = {}
feature_cols_per_step = {}

for step in range(1, FORECAST_POINTS + 1):
    target_col = f'target_step_{step}'
    print(f'\n=== Step {step} ===')

    df = build_features(train_df, step=step)
    df = df.dropna(subset=[target_col])

    feature_cols = [c for c in df.columns if c not in exclude_base]
    cat_features = [c for c in cat_features_base if c in feature_cols]
    feature_cols_per_step[step] = feature_cols

    # Bias correction на последних 3 днях
    val_start = df['timestamp'].max() - pd.Timedelta(days=3)
    val_df    = df[df['timestamp'] >= val_start]
    fit_df    = df[df['timestamp'] <  val_start]

    model_val = lgb.LGBMRegressor(**LGBM_PARAMS)
    model_val.fit(fit_df[feature_cols], fit_df[target_col], categorical_feature=cat_features)
    val_pred = model_val.predict(val_df[feature_cols]).clip(0)
    bias = val_pred.sum() / val_df[target_col].values.sum() - 1
    bias_factors[step] = bias

    val_pred_bc = (val_pred / (1 + bias)).clip(0)
    y_true = val_df[target_col].values
    wape  = np.abs(val_pred_bc - y_true).sum() / y_true.sum()
    rbias = abs(val_pred_bc.sum() / y_true.sum() - 1)
    print(f'  val={wape+rbias:.4f}  bias={bias:+.4f}')

    # Обучаем на всём train
    model_full = lgb.LGBMRegressor(**LGBM_PARAMS)
    model_full.fit(df[feature_cols], df[target_col], categorical_feature=cat_features)
    models[step] = model_full

    del df, fit_df, val_df, model_val; gc.collect()

print('\nВсе модели обучены!')

joblib.dump({
    'models': models,
    'bias_factors': bias_factors,
    'feature_cols_per_step': feature_cols_per_step,
}, 'models_v3.pkl')
print('Сохранено: models_v3.pkl')


=== Step 1 ===
  val=0.2526  bias=-0.0326

=== Step 2 ===
  val=0.3184  bias=-0.0560

=== Step 3 ===
  val=0.3318  bias=-0.0470

=== Step 4 ===
  val=0.3368  bias=-0.0357

=== Step 5 ===
  val=0.3399  bias=-0.0330

=== Step 6 ===
  val=0.3408  bias=-0.0306

=== Step 7 ===
  val=0.3417  bias=-0.0280

=== Step 8 ===
  val=0.3421  bias=-0.0230

Все модели обучены!
Сохранено: models_v3.pkl


In [6]:
# ── Ансамбль по последним точкам ─────────────────────────────────────────────
from collections import defaultdict

lag_tail = train_df.groupby('route_id', sort=False).tail(700)
lag_tail  = lag_tail.sort_values(['route_id', 'timestamp']).reset_index(drop=True)

last_ts_per_route = train_df.groupby('route_id')['timestamp'].max()

weighted_preds = defaultdict(lambda: defaultdict(float))
weight_sums    = defaultdict(lambda: defaultdict(float))

for offset, weight in zip(ENSEMBLE_OFFSETS, ENSEMBLE_WEIGHTS):
    print(f'\n--- Offset={offset}, weight={weight} ---')
    for test_step in range(1, FORECAST_POINTS + 1):
        effective_step = test_step + offset
        if effective_step > FORECAST_POINTS:
            continue

        feature_cols = feature_cols_per_step[effective_step]
        df_feat = build_features(lag_tail, step=effective_step)
        df_src  = df_feat.groupby('route_id', sort=False).nth(-(offset + 1)).reset_index()

        pred    = models[effective_step].predict(df_src[feature_cols]).clip(0)
        pred_bc = (pred / (1 + bias_factors[effective_step])).clip(0)

        for i, row in df_src.iterrows():
            rid = row['route_id']
            weighted_preds[rid][test_step] += weight * pred_bc[i]
            weight_sums[rid][test_step]    += weight

        print(f'  test_step={test_step} (model={effective_step}): done')
        del df_feat, df_src; gc.collect()

print('\nАнсамбль готов')


--- Offset=0, weight=0.4 ---
  test_step=1 (model=1): done
  test_step=2 (model=2): done
  test_step=3 (model=3): done
  test_step=4 (model=4): done
  test_step=5 (model=5): done
  test_step=6 (model=6): done
  test_step=7 (model=7): done
  test_step=8 (model=8): done

--- Offset=1, weight=0.3 ---
  test_step=1 (model=2): done
  test_step=2 (model=3): done
  test_step=3 (model=4): done
  test_step=4 (model=5): done
  test_step=5 (model=6): done
  test_step=6 (model=7): done
  test_step=7 (model=8): done

--- Offset=2, weight=0.2 ---
  test_step=1 (model=3): done
  test_step=2 (model=4): done
  test_step=3 (model=5): done
  test_step=4 (model=6): done
  test_step=5 (model=7): done
  test_step=6 (model=8): done

--- Offset=3, weight=0.1 ---
  test_step=1 (model=4): done
  test_step=2 (model=5): done
  test_step=3 (model=6): done
  test_step=4 (model=7): done
  test_step=5 (model=8): done

Ансамбль готов


In [7]:
# ── Сабмит ────────────────────────────────────────────────────────────────────
records = []
for rid in last_ts_per_route.index:
    last_ts = last_ts_per_route[rid]
    for test_step in range(1, FORECAST_POINTS + 1):
        pred_ts = last_ts + pd.Timedelta(minutes=30 * test_step)
        w_sum   = weight_sums[rid][test_step]
        y_pred  = weighted_preds[rid][test_step] / w_sum if w_sum > 0 else 0.0
        records.append({'route_id': rid, 'timestamp': pred_ts, 'y_pred': y_pred})

pred_long = pd.DataFrame(records)
pred_long['timestamp'] = pd.to_datetime(pred_long['timestamp'])
test_df['timestamp']   = pd.to_datetime(test_df['timestamp'])

submission = test_df.merge(pred_long, on=['route_id', 'timestamp'], how='left')
submission['y_pred'] = submission['y_pred'].fillna(0).clip(lower=0)

submission[['id', 'y_pred']].to_csv(SUBMISSION_PATH, index=False)
print(f'Сохранено: {SUBMISSION_PATH}')
print(submission[['id', 'route_id', 'timestamp', 'y_pred']].head(10))
print('NaN:', submission['y_pred'].isna().sum())
print('y_pred mean:', submission['y_pred'].mean())

Сохранено: submission_v7.csv
   id  route_id           timestamp      y_pred
0   0       471 2025-11-01 11:00:00 123635.2338
1   1       471 2025-11-01 11:30:00 140929.2106
2   2       471 2025-11-01 12:00:00 156245.5017
3   3       471 2025-11-01 12:30:00 149730.1210
4   4       471 2025-11-01 13:00:00 151254.9255
5   5       471 2025-11-01 13:30:00 151673.2657
6   6       471 2025-11-01 14:00:00 168030.2544
7   7       471 2025-11-01 14:30:00 179556.5691
8   8       496 2025-11-01 11:00:00 271167.5477
9   9       496 2025-11-01 11:30:00 263535.9253
NaN: 0
y_pred mean: 278387.7060526153


In [6]:
train_df.describe()

,route_id,timestamp,status_1,status_2,status_3,status_4,status_5,status_6,target_1h
count,4630000.0000,4630000,4630000.0000,4630000.0000,4630000.0000,4630000.0000,4630000.0000,4630000.0000,4630000.0000
mean,499.5000,2025-09-14 05:14:59.999999488,22.6768,22.8585,22.8568,119215.6315,701978.8944,3344.6073,261242.1379
min,0.0000,2025-07-28 00:00:00,0.0000,0.0000,0.0000,1390.0000,18519.0000,0.0000,0.0000
25%,249.7500,2025-08-21 02:30:00,13.0000,13.0000,8.0000,74967.0000,389779.0000,46.0000,138990.0000
50%,499.5000,2025-09-14 05:15:00,21.0000,21.0000,19.0000,109912.0000,618836.0000,162.0000,235769.0000
75%,749.2500,2025-10-08 08:00:00,30.0000,30.0000,33.0000,152309.0000,915522.0000,1243.0000,355175.0000
max,999.0000,2025-11-01 10:30:00,802.0000,522.0000,620.0000,587938.0000,3503114.0000,152739.0000,10831337.0000
std,288.6750,NaN,13.0414,12.8578,19.4400,59825.6547,421306.3096,10048.1682,169082.5867
